# Medical guideline RAG: evidence-first ingestion and retrieval

This notebook builds the **Day-1 retrieval foundation** for a clinical decision-support prototype. It ingests official asthma guidelines, preserves page and section provenance, creates section-aware chunks, embeds them with Gemini, stores them in Chroma, and displays retrieved evidence for clinical test queries.

**Clinical scope:** asthma diagnosis and guideline-based management in children and adolescents. The scope is deliberately narrow enough to evaluate retrieval quality across two complementary public sources.

> **Safety boundary:** This is an educational retrieval prototype, not a diagnostic system and not medical advice. It does not generate treatment recommendations. A later generation layer must cite retrieved evidence, refuse when evidence is insufficient, and preserve clinician judgment.

**Core principle:** A fluent answer is not necessarily a safe answer. Every eventual clinical claim must trace to an official guideline, section, page, and chunk.

## 1. Lab requirements and definition of done

The implementation follows `Docs/Day-1.pdf`, `Docs/AI AGEDNDA.pdf`, and the hands-on lab slide:

1. Choose one narrow clinical topic.
2. Use 1–2 official, publicly accessible guideline PDFs and record why each source is credible and usable.
3. Extract text page by page and inspect samples before indexing.
4. Clean conservatively: remove recurring layout noise without deleting clinical values, dates, grades, or recommendations.
5. Create section-aware chunks in the recommended 400–800-token range with overlap inside section boundaries.
6. Attach `document`, `section`, `page`, `chunk_id`, and source provenance to every chunk.
7. Generate embeddings and build a persistent vector index.
8. Run 5–10 clinical queries and display the retrieved chunks, scores, and metadata before any answer-generation work.

**Stop point:** Do not optimize prompts or build a UI until this first index retrieves reasonable, inspectable evidence.

## 2. Environment setup

The original notebook installed overlapping packages twice and omitted `langchain-google-genai`, even though it imported the Gemini embedding class. The single cell below installs only the libraries used here.

- `pymupdf`: page-preserving PDF extraction
- `tiktoken`: repeatable token counts for chunk sizing
- `langchain-core`: the `Document` data model
- `langchain-chroma` and `chromadb`: persistent vector storage
- `langchain-google-genai`: Gemini embeddings
- `pandas`: compact quality-control tables

Run this once per fresh Colab runtime, then restart the runtime only if the installer asks you to.

In [ ]:
%pip install -qU pymupdf tiktoken langchain-core langchain-chroma chromadb langchain-google-genai pandas

## 3. Imports and reproducible configuration

All tunable values live in one place. A 700-token target with 100-token overlap stays inside the course recommendation of 400–800 tokens while retaining enough local context for guideline recommendations. Overlap is applied **only within the same detected section**, preventing one section from being mislabeled as another.

The notebook works both from this repository and in Google Colab. In Colab, missing PDFs are uploaded interactively; locally, the existing `Docs/Sources` directory is discovered automatically.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import sys
from collections import Counter
from getpass import getpass
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
import pymupdf
import tiktoken
from IPython.display import display
from langchain_core.documents import Document


RUNNING_IN_COLAB = "google.colab" in sys.modules
WORKING_DIR = Path.cwd()

source_candidates = [
    WORKING_DIR / "Docs" / "Sources",
    WORKING_DIR.parent / "Docs" / "Sources",
    Path("/content/guidelines"),
]
SOURCE_DIR = next((p for p in source_candidates if p.exists()), source_candidates[-1])
ARTIFACT_DIR = (Path("/content/medical_rag_artifacts") if RUNNING_IN_COLAB
                else WORKING_DIR / "medical_rag_artifacts")
PERSIST_DIR = ARTIFACT_DIR / "chroma_db"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CLINICAL_SCOPE = "Asthma diagnosis and guideline-based management in children and adolescents"
SELECTED_FILENAMES = ["WHO asthma.pdf", "NICE Asthma.pdf"]
CHUNK_SIZE = 700
CHUNK_OVERLAP = 100
MIN_CHUNK_TOKENS = 80
TOP_K = 5
EMBEDDING_MODEL = "models/gemini-embedding-001"
COLLECTION_PREFIX = "pediatric_asthma_guidelines"

assert 400 <= CHUNK_SIZE <= 800
assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE

print(f"Runtime: {'Google Colab' if RUNNING_IN_COLAB else 'local Jupyter'}")
print(f"Source directory: {SOURCE_DIR.resolve() if SOURCE_DIR.exists() else SOURCE_DIR}")
print(f"Artifact directory: {ARTIFACT_DIR.resolve()}")

## 4. Scope, source selection, credibility, and public usability

Only two PDFs are selected, matching the lab constraint. Both are official guideline publishers rather than blogs or secondary summaries:

| File | Credibility | Public/legal usability recorded for this prototype |
|---|---|---|
| `WHO asthma.pdf` | Published by the World Health Organization as *WHO consolidated guidelines for the management of common childhood illness: management of asthma in children and adolescents and bronchiolitis in infants and young children* (2026). | The PDF states on page 4 that it is available under **CC BY-NC-SA 3.0 IGO**. Retain attribution and review the license before any commercial reuse. |
| `NICE Asthma.pdf` | NICE guideline NG245, *Asthma: diagnosis, monitoring and chronic asthma management (BTS, NICE, SIGN)*, published 27 November 2024 and updated in the supplied 2026 copy. | Publicly accessible from the official NICE guidance portal. Copyright and notice-of-rights terms still apply; public access is not the same as unrestricted relicensing. |

The registry below is copied into every chunk so provenance survives retrieval. If a different edition is supplied, update the registry and re-check its publication and reuse terms.

In [ ]:
SOURCE_REGISTRY = {
    "WHO asthma.pdf": {
        "publisher": "World Health Organization (WHO)",
        "source_url": "https://iris.who.int/",
        "credibility": "Official WHO consolidated clinical guideline (2026).",
        "public_access": "Public WHO publication; supplied PDF states CC BY-NC-SA 3.0 IGO on page 4.",
    },
    "NICE Asthma.pdf": {
        "publisher": "National Institute for Health and Care Excellence (NICE)",
        "source_url": "https://www.nice.org.uk/guidance/ng245",
        "credibility": "Official NICE guideline NG245, developed with BTS and SIGN.",
        "public_access": "Public official guidance; reuse remains subject to NICE notice-of-rights terms.",
    },
}


def resolve_source_paths() -> list[Path]:
    """Find the configured PDFs locally or request missing files in Colab."""
    SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    missing = [name for name in SELECTED_FILENAMES if not (SOURCE_DIR / name).exists()]

    if missing and RUNNING_IN_COLAB:
        from google.colab import files

        print("Upload the missing official guideline PDFs:", ", ".join(missing))
        uploaded = files.upload()
        for uploaded_name, content in uploaded.items():
            destination = SOURCE_DIR / Path(uploaded_name).name
            destination.write_bytes(content)

    paths = [SOURCE_DIR / name for name in SELECTED_FILENAMES]
    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing configured guideline PDF(s): " + ", ".join(missing)
            + ". Add the files or change SELECTED_FILENAMES."
        )

    unregistered = [path.name for path in paths if path.name not in SOURCE_REGISTRY]
    if unregistered:
        raise KeyError(f"Add source provenance to SOURCE_REGISTRY for: {unregistered}")
    return paths


source_paths = resolve_source_paths()
display(pd.DataFrame([
    {"file": path.name, **SOURCE_REGISTRY[path.name]} for path in source_paths
]))

## 5. Page-preserving extraction and conservative cleaning

Clinical text cleaning should be auditable and deliberately modest. Aggressive regex removal can silently delete medication doses, dates, recommendation grades, or symbols.

This cleaner therefore:

- extracts each PDF page independently so printed-page provenance is never inferred later;
- detects recurring running headers and footers only in the first/last three non-empty lines;
- removes page-number-only lines and table-of-contents dot leaders;
- repairs simple word hyphenation across line breaks and normalizes whitespace;
- protects dosage-like expressions, dates, evidence levels, and recommendation labels from recurring-line removal;
- records every removed line with page and reason for inspection.

It does **not** remove URLs, references, clinical units, arrows, or copyright/license text globally. Those may carry provenance or clinical meaning.

In [ ]:
ZONE_LINES = 3
MAX_RUNNING_LINE_LENGTH = 140

PAGE_NUMBER_RE = re.compile(
    r"^(?:page\s+)?\d{1,4}(?:\s*(?:of|/)\s*\d{1,4})?$", re.IGNORECASE
)
TOC_LEADER_RE = re.compile(r"^.{2,160}?(?:\.{4,}|\s\.{3,})\s*[ivxlcdm\d-]{1,8}$", re.IGNORECASE)
PROTECTED_CLINICAL_RE = re.compile(
    r"(?:\b\d+(?:\.\d+)?\s*(?:mg|mcg|g|kg|mL|L|IU|units?|mmol|mEq|mg/dL|"
    r"mg/kg|mmHg|bpm|%)\b|\brecommendation\s+\d+[a-z]?\b|\bgrade\s+[A-D]\b|"
    r"\blevel\s+(?:I{1,3}|IV|V)\b|\b\d{4}-\d{2}-\d{2}\b|\b\d{4}\b)",
    re.IGNORECASE,
)


def extract_raw_pages(pdf_path: Path) -> list[dict[str, Any]]:
    """Return one record per physical PDF page using 1-based page numbers."""
    pages = []
    with pymupdf.open(pdf_path) as pdf:
        if pdf.needs_pass:
            raise ValueError(f"Encrypted PDF requires a password: {pdf_path.name}")
        for page_index, page in enumerate(pdf):
            pages.append({
                "page": page_index + 1,
                "raw_text": page.get_text("text", sort=True) or "",
            })
    if not pages:
        raise ValueError(f"No pages were extracted from {pdf_path.name}")
    return pages


def nonempty_line_indexes(text: str) -> list[int]:
    return [i for i, line in enumerate(text.splitlines()) if line.strip()]


def zone_line_indexes(text: str) -> set[int]:
    indexes = nonempty_line_indexes(text)
    return set(indexes[:ZONE_LINES] + indexes[-ZONE_LINES:])


def normalize_running_line(line: str) -> str:
    normalized = re.sub(r"\s+", " ", line.strip().lower())
    normalized = re.sub(r"\b\d+\b", "#", normalized)
    return normalized


def find_recurring_running_lines(
    pages: list[dict[str, Any]], min_ratio: float = 0.35
) -> set[str]:
    """Detect normalized header/footer candidates without scanning body text."""
    if len(pages) < 4:
        return set()
    counts: Counter[str] = Counter()
    for page in pages:
        lines = page["raw_text"].splitlines()
        seen_on_page = set()
        for index in zone_line_indexes(page["raw_text"]):
            line = lines[index].strip()
            if (
                not line
                or len(line) > MAX_RUNNING_LINE_LENGTH
                or PROTECTED_CLINICAL_RE.search(line)
            ):
                continue
            normalized = normalize_running_line(line)
            if normalized not in seen_on_page:
                counts[normalized] += 1
                seen_on_page.add(normalized)
    threshold = max(3, round(len(pages) * min_ratio))
    return {line for line, count in counts.items() if count >= threshold}


def clean_page(
    raw_text: str, page_number: int, recurring_lines: set[str]
) -> tuple[str, list[dict[str, Any]]]:
    lines = raw_text.replace("\u00ad", "").replace("\x00", "").splitlines()
    zone_indexes = zone_line_indexes(raw_text)
    kept = []
    audit = []

    for index, original in enumerate(lines):
        line = re.sub(r"[ \t]+", " ", original).strip()
        if not line:
            kept.append("")
            continue

        reason = None
        if index in zone_indexes and PAGE_NUMBER_RE.fullmatch(line):
            reason = "page_number"
        elif (
            index in zone_indexes
            and normalize_running_line(line) in recurring_lines
            and not PROTECTED_CLINICAL_RE.search(line)
        ):
            reason = "recurring_header_or_footer"
        elif TOC_LEADER_RE.fullmatch(line):
            reason = "table_of_contents_leader"

        if reason:
            audit.append({"page": page_number, "reason": reason, "text": line})
        else:
            kept.append(line)

    text = "\n".join(kept)
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip(), audit


def parse_guideline(pdf_path: Path) -> dict[str, Any]:
    raw_pages = extract_raw_pages(pdf_path)
    recurring = find_recurring_running_lines(raw_pages)
    cleaned_pages = []
    removal_audit = []

    for page in raw_pages:
        cleaned_text, audit = clean_page(page["raw_text"], page["page"], recurring)
        cleaned_pages.append({
            "page": page["page"],
            "raw_text": page["raw_text"],
            "cleaned_text": cleaned_text,
            "raw_chars": len(page["raw_text"]),
            "cleaned_chars": len(cleaned_text),
        })
        removal_audit.extend(audit)

    return {
        "document": pdf_path.stem,
        "file_name": pdf_path.name,
        "file_path": str(pdf_path.resolve()),
        "page_count": len(cleaned_pages),
        "recurring_lines": sorted(recurring),
        "pages": cleaned_pages,
        "removal_audit": removal_audit,
    }

## 6. Extract, audit, and inspect sample pages

Never index a PDF immediately after extraction. The output below checks page counts, character retention, recurring-line decisions, and representative raw/cleaned text. Empty or garbled pages can indicate scanned images, unusual fonts, or a need for OCR/layout-aware parsing.

For each source, inspect at least the title/copyright pages and one recommendation page. The automatic preview chooses the first substantial page; adjust `pages_to_preview` if a known recommendation page should be inspected explicitly.

In [ ]:
parsed_documents = [parse_guideline(path) for path in source_paths]

summary_rows = []
for document in parsed_documents:
    total_raw = sum(page["raw_chars"] for page in document["pages"])
    total_cleaned = sum(page["cleaned_chars"] for page in document["pages"])
    summary_rows.append({
        "document": document["document"],
        "pages": document["page_count"],
        "raw_characters": total_raw,
        "cleaned_characters": total_cleaned,
        "characters_retained_pct": round(100 * total_cleaned / max(total_raw, 1), 1),
        "removed_lines": len(document["removal_audit"]),
        "recurring_patterns": len(document["recurring_lines"]),
    })

display(pd.DataFrame(summary_rows))

for document in parsed_documents:
    substantial = [p for p in document["pages"] if len(p["cleaned_text"]) >= 500]
    pages_to_preview = document["pages"][:1] + substantial[:1]
    print("\n" + "=" * 100)
    print(f"DOCUMENT: {document['document']}")
    print("Detected recurring header/footer patterns:", document["recurring_lines"][:10])
    for page in pages_to_preview:
        print(f"\n--- Cleaned PDF page {page['page']} ---")
        print(page["cleaned_text"][:1800] or "[No extractable text]")

removal_rows = [
    {"document": doc["document"], **item}
    for doc in parsed_documents
    for item in doc["removal_audit"]
]
display(pd.DataFrame(removal_rows).head(30) if removal_rows else pd.DataFrame(
    columns=["document", "page", "reason", "text"]
))

## 7. Section-aware chunking and complete metadata

Chunking is performed over line/paragraph units while carrying their physical PDF page numbers. A new detected heading closes the previous section before any overlap is considered. This prevents text from section A being repeated into a chunk labeled as section B.

Long paragraphs are token-split safely. Stable chunk IDs combine the document slug, sequence number, and a short content hash, so IDs remain unique across documents and change when the underlying content changes.

Every LangChain `Document` contains these scalar Chroma-compatible metadata fields:

- `document`, `publisher`, `source_url`, and `topic`
- `section`
- `page` (citation-friendly starting page), plus `page_start` and `page_end`
- `chunk_id`
- `token_count`

In [ ]:
ENCODER = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    return len(ENCODER.encode(text))


def is_section_heading(line: str) -> bool:
    """Conservative heuristic for common guideline heading styles."""
    text = re.sub(r"\s+", " ", line.strip())
    if not text or len(text) > 180 or TOC_LEADER_RE.fullmatch(text):
        return False
    if re.match(r"^(?:\d+(?:\.\d+)*\.?|section\s+\d+|chapter\s+\d+|part\s+[ivx\d]+)\s+\S", text, re.I):
        return True
    if re.match(r"^(?:recommendation|recommendations|appendix|annex)\b", text, re.I):
        return True
    if text.isupper() and 4 <= len(text) <= 100 and not text.endswith(('.', ';')):
        return True
    clinical_heading_terms = (
        "diagnosis", "management", "monitoring", "treatment", "recommendation",
        "assessment", "follow-up", "follow up", "evidence", "scope", "introduction",
    )
    word_count = len(text.split())
    return (
        word_count <= 14
        and not text.endswith(('.', ';'))
        and any(term in text.lower() for term in clinical_heading_terms)
    )


def split_oversized_unit(text: str, page: int, section: str) -> list[dict[str, Any]]:
    tokens = ENCODER.encode(text)
    if len(tokens) <= CHUNK_SIZE:
        return [{"text": text, "page": page, "section": section, "tokens": len(tokens)}]
    return [
        {
            "text": ENCODER.decode(tokens[start:start + CHUNK_SIZE]),
            "page": page,
            "section": section,
            "tokens": len(tokens[start:start + CHUNK_SIZE]),
        }
        for start in range(0, len(tokens), CHUNK_SIZE)
    ]


def overlap_tail(units: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Return up to CHUNK_OVERLAP tokens from the same section."""
    if not units or CHUNK_OVERLAP == 0:
        return []
    remaining = CHUNK_OVERLAP
    tail = []
    for unit in reversed(units):
        if remaining <= 0:
            break
        if unit["tokens"] <= remaining:
            tail.insert(0, dict(unit))
            remaining -= unit["tokens"]
        else:
            tokens = ENCODER.encode(unit["text"])[-remaining:]
            tail.insert(0, {
                **unit,
                "text": ENCODER.decode(tokens),
                "tokens": len(tokens),
            })
            remaining = 0
    return tail


def document_slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")[:48]


def make_chunk(
    units: list[dict[str, Any]], document: dict[str, Any], sequence: int
) -> Document | None:
    text = "\n".join(unit["text"] for unit in units).strip()
    if not text:
        return None
    pages = [unit["page"] for unit in units]
    section = units[0]["section"]
    digest = hashlib.sha1(text.encode("utf-8")).hexdigest()[:10]
    chunk_id = f"{document_slug(document['document'])}-{sequence:04d}-{digest}"
    registry = SOURCE_REGISTRY[document["file_name"]]
    metadata = {
        "document": document["document"],
        "publisher": registry["publisher"],
        "source_url": registry["source_url"],
        "topic": CLINICAL_SCOPE,
        "section": section,
        "page": int(min(pages)),
        "page_start": int(min(pages)),
        "page_end": int(max(pages)),
        "chunk_id": chunk_id,
        "token_count": int(count_tokens(text)),
    }
    return Document(page_content=text, metadata=metadata)


def chunk_guideline(document: dict[str, Any]) -> list[Document]:
    chunks = []
    current_section = "Front matter / section not yet detected"
    buffer = []
    sequence = 1

    def flush(keep_overlap: bool) -> None:
        nonlocal buffer, sequence
        chunk = make_chunk(buffer, document, sequence)
        if chunk is not None:
            chunks.append(chunk)
            sequence += 1
        buffer = overlap_tail(buffer) if keep_overlap else []

    for page in document["pages"]:
        for raw_unit in re.split(r"\n{1,}", page["cleaned_text"]):
            unit_text = raw_unit.strip()
            if not unit_text:
                continue

            heading = is_section_heading(unit_text)
            if heading and unit_text != current_section:
                if buffer:
                    flush(keep_overlap=False)
                current_section = unit_text

            for unit in split_oversized_unit(unit_text, page["page"], current_section):
                buffered_tokens = sum(item["tokens"] for item in buffer)
                section_changed = bool(buffer and buffer[0]["section"] != unit["section"])
                would_overflow = bool(buffer and buffered_tokens + unit["tokens"] > CHUNK_SIZE)
                if section_changed:
                    flush(keep_overlap=False)
                elif would_overflow:
                    flush(keep_overlap=True)
                buffer.append(unit)

    if buffer:
        flush(keep_overlap=False)

    # Merge a tiny final chunk into its same-section predecessor when safe.
    if len(chunks) >= 2 and chunks[-1].metadata["token_count"] < MIN_CHUNK_TOKENS:
        previous, final = chunks[-2], chunks[-1]
        if previous.metadata["section"] == final.metadata["section"]:
            merged_text = previous.page_content + "\n" + final.page_content
            if count_tokens(merged_text) <= CHUNK_SIZE + CHUNK_OVERLAP:
                merged_units = [{
                    "text": merged_text,
                    "page": previous.metadata["page_start"],
                    "section": previous.metadata["section"],
                    "tokens": count_tokens(merged_text),
                }]
                merged = make_chunk(merged_units, document, len(chunks) - 1)
                merged.metadata["page_end"] = final.metadata["page_end"]
                chunks[-2:] = [merged]
    return chunks

## 8. Create chunks, save an audit manifest, and validate invariants

The manifest stores chunk text beside metadata for human review. The assertions are intentional: a vector index with missing provenance is considered a failed build, even if embedding succeeds.

The token-size distribution is also shown. A few chunks may be shorter at strict section boundaries; preserving the recommendation boundary is usually more important than padding unrelated material into the chunk.

In [ ]:
chunks = [
    chunk
    for parsed_document in parsed_documents
    for chunk in chunk_guideline(parsed_document)
]

if not chunks:
    raise RuntimeError("No chunks were created. Inspect the extraction previews before continuing.")

REQUIRED_METADATA = {
    "document", "publisher", "source_url", "topic", "section", "page",
    "page_start", "page_end", "chunk_id", "token_count",
}

chunk_ids = [chunk.metadata["chunk_id"] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Chunk IDs must be globally unique."

for chunk in chunks:
    missing = REQUIRED_METADATA - set(chunk.metadata)
    assert not missing, f"{chunk.metadata.get('chunk_id')} missing metadata: {missing}"
    assert chunk.page_content.strip(), "Empty chunks cannot be indexed."
    assert chunk.metadata["page_start"] <= chunk.metadata["page_end"]
    assert chunk.metadata["page"] == chunk.metadata["page_start"]

manifest = [
    {"text": chunk.page_content, "metadata": chunk.metadata}
    for chunk in chunks
]
manifest_path = ARTIFACT_DIR / "chunks_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

chunk_table = pd.DataFrame([chunk.metadata for chunk in chunks])
display(chunk_table.groupby("document").agg(
    chunks=("chunk_id", "count"),
    min_tokens=("token_count", "min"),
    median_tokens=("token_count", "median"),
    max_tokens=("token_count", "max"),
    first_page=("page_start", "min"),
    last_page=("page_end", "max"),
))
display(chunk_table.head(10))

oversized = chunk_table[chunk_table["token_count"] > CHUNK_SIZE + CHUNK_OVERLAP]
assert oversized.empty, f"Unexpected oversized chunks:\n{oversized[['chunk_id', 'token_count']]}"
print(f"Validated {len(chunks)} chunks. Audit manifest: {manifest_path}")

## 9. Gemini embeddings and persistent Chroma index

The API key is requested with `getpass`, so it is not displayed in notebook output or saved in the notebook. Do not hard-code or commit credentials.

The collection name includes a corpus fingerprint. Changing the selected documents or chunk contents creates a new collection instead of silently mixing incompatible experiments. Documents are added in batches to make API failures easier to diagnose and retry.

Chroma persists the index under `medical_rag_artifacts/chroma_db`. Treat the index as derived guideline data: preserve the source licenses and do not publish it automatically.

In [ ]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings


if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI API key: ").strip()
if not os.environ["GOOGLE_API_KEY"]:
    raise ValueError("GOOGLE_API_KEY is required to create Gemini embeddings.")

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
corpus_fingerprint = hashlib.sha1(
    "|".join(chunk.metadata["chunk_id"] for chunk in chunks).encode("utf-8")
).hexdigest()[:10]
collection_name = f"{COLLECTION_PREFIX}_{corpus_fingerprint}"

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(PERSIST_DIR),
    collection_metadata={"hnsw:space": "cosine"},
)

BATCH_SIZE = 32
for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    vector_store.add_documents(
        documents=batch,
        ids=[document.metadata["chunk_id"] for document in batch],
    )
    print(f"Indexed {min(start + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks")

print(f"Collection: {collection_name}")
print(f"Persistent index: {PERSIST_DIR}")

## 10. Evidence-panel retrieval test: eight queries

This is retrieval evaluation, not answer generation. For every query the evidence panel prints rank, relevance score, document, section, physical PDF page range, chunk ID, source URL, and chunk text.

The eight queries cover acute and long-term asthma management, diagnosis/monitoring, and one deliberately out-of-scope diabetes question. The out-of-scope query is useful because a vector store always returns nearest neighbors; a returned chunk is **not proof that the evidence is sufficient**. Score thresholds must be calibrated later on labeled data rather than guessed here.

`TOP_K = 5` follows the course deck’s balanced starting point. Day 2 should compare top-3/top-5/top-10, chunk sizes, semantic versus keyword/hybrid search, and reranking using 15–20 labeled questions and Precision@k.

In [ ]:
TEST_QUERIES = [
    "What additional treatments are recommended with standard first-line therapy for acute asthma exacerbations in children?",
    "When is intravenous magnesium sulfate considered for a child with an acute asthma exacerbation?",
    "What second-line therapy options are recommended for paediatric asthma exacerbations?",
    "What does the guideline recommend for long-term asthma management in children and adolescents?",
    "Which objective tests are used to diagnose asthma in children and young people?",
    "How are FeNO and spirometry used when diagnosing asthma?",
    "How should asthma control be monitored during follow-up?",
    "What is the first-line drug treatment for type 2 diabetes in adults?",  # deliberately out of scope
]


def retrieve_evidence(query: str, top_k: int = TOP_K) -> list[dict[str, Any]]:
    results = vector_store.similarity_search_with_relevance_scores(query, k=top_k)
    return [
        {
            "rank": rank,
            "score": float(score),
            "text": document.page_content,
            "metadata": dict(document.metadata),
        }
        for rank, (document, score) in enumerate(results, start=1)
    ]


retrieval_log = []
for query in TEST_QUERIES:
    evidence = retrieve_evidence(query)
    retrieval_log.append({"query": query, "results": evidence})
    print("\n" + "=" * 110)
    print("QUERY:", query)
    for result in evidence:
        meta = result["metadata"]
        page_label = (
            str(meta["page_start"]) if meta["page_start"] == meta["page_end"]
            else f"{meta['page_start']}-{meta['page_end']}"
        )
        print("-" * 110)
        print(
            f"Rank {result['rank']} | score={result['score']:.4f} | "
            f"{meta['document']} | section={meta['section']} | PDF page(s)={page_label}"
        )
        print(f"chunk_id={meta['chunk_id']} | source={meta['source_url']}")
        print(result["text"][:1400].strip())

retrieval_log_path = ARTIFACT_DIR / "retrieval_log.json"
retrieval_log_path.write_text(
    json.dumps(retrieval_log, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"\nSaved retrieval evidence log: {retrieval_log_path}")

## 11. Human retrieval audit and handoff

Automated similarity scores do not establish clinical relevance. Review the displayed evidence and fill in the audit table below:

- `relevant_at_k`: number of top-k chunks that directly support the information need;
- `best_chunk_ids`: IDs of the strongest supporting chunks;
- `citation_metadata_correct`: whether document, section, and physical PDF page match the source;
- `notes`: missing evidence, noisy chunks, duplicates, or misleading retrievals.

For a labeled query with a known set of relevant chunks, compute `Precision@k = relevant chunks in top-k / k`. At least 15–20 labeled clinical questions are recommended for Day-2 comparison; these eight are the Day-1 smoke test requested in the lab.

In [ ]:
audit_template = pd.DataFrame([
    {
        "query": item["query"],
        "top_k": len(item["results"]),
        "top_score": round(item["results"][0]["score"], 4) if item["results"] else None,
        "retrieved_documents": ", ".join(sorted({
            result["metadata"]["document"] for result in item["results"]
        })),
        "relevant_at_k": "",
        "best_chunk_ids": "",
        "citation_metadata_correct": "",
        "notes": "",
    }
    for item in retrieval_log
])

audit_path = ARTIFACT_DIR / "retrieval_audit.csv"
audit_template.to_csv(audit_path, index=False)
display(audit_template)
print(f"Fill in and retain this evaluation artifact: {audit_path}")

## 12. What is complete, what is intentionally deferred, and known limitations

### Day-1 completion checklist

- Narrow pediatric/adolescent asthma scope is declared.
- Two official public guideline PDFs and their reuse context are documented.
- Physical pages are extracted, conservatively cleaned, sampled, and audited.
- Section boundaries and 700/100 token chunking are implemented.
- Every chunk has document, publisher, URL, section, physical page range, stable ID, topic, and token count.
- Chunks are embedded with Gemini and persisted in Chroma.
- Eight queries display retrieved evidence and complete provenance before generation.
- Retrieval logs and a human evaluation template are saved.

### Intentionally deferred

Prompt optimization, answer generation, UI work, confidence thresholds, unsupported-claim detection, and refusal behavior belong after retrieval quality is measured. Later stages must produce structured output with recommendation, quoted/paraphrased supporting evidence, document/section/page/chunk citation, confidence, and an insufficient-evidence path.

### Known limitations to inspect

- Heading detection is heuristic; tables, multi-column pages, and unusual typography may require layout-aware parsing.
- PDF page numbers here mean physical PDF pages, which can differ from printed page labels. If printed labels are required, capture both explicitly.
- Image-only/scanned pages need OCR; this notebook raises attention through empty/low-text previews but does not OCR automatically.
- Semantic similarity is not calibrated clinical confidence. The out-of-scope query is expected to return neighbors and must not be treated as supported advice.
- Source updates require re-verifying edition, publication date, license/terms, and the source registry before re-indexing.